# Trajectory quantification and plotting

These demos target **movement 0.17.0** and one explicitly selected session.
`data-conduit` selects, reads and aligns the experiment's DLC and event streams;
`movement` filters the pose and calculates the measurements. Matplotlib arranges
panels and the shared template adds event labels.

The time axis remains the recorded session clock in **seconds**. Spatial values
remain **pixels**; no arena calibration is inferred. Filtering settings are
examples to review for this session, not validated paper analysis settings.
Processing runs on the continuous session before any trial is sliced.
No real-data output has been generated in this template.

In [ ]:
from pathlib import Path
import sys

# Locate this checkout when Jupyter starts in a notebook subdirectory.
for parent in (Path.cwd(), *Path.cwd().parents):
    if (parent / "movement_figures").is_dir():
        sys.path.insert(0, str(parent))
        break
    if (parent / "src" / "movement_figures").is_dir():
        sys.path.insert(0, str(parent / "src"))
        break
else:
    raise RuntimeError("Start Jupyter inside the data-conduit checkout.")

import matplotlib.pyplot as plt
import movement
import movement.kinematics as kin
import numpy as np
import pandas as pd
from IPython.display import display
from movement.plots import plot_centroid_trajectory, plot_occupancy
from movement.utils.vector import compute_signed_angle_2d

from movement_figures.data_template.loading import (
    build_config, load_session, prepare_pose, preview_session,
)
from movement_figures.timeseries_template.annotations import annotate_qc_timeseries, qc_annotations

print(f"movement {movement.__version__}: {movement.__file__}")

## Session and processing settings

In [ ]:
# Edit this cell for your data. The same settings apply to all three notebooks.
ROOT = Path("/path/to/Training")
SESSION = "replace-with-one-session-folder-name"
LEVEL_NAMES = ("mouseID", "day")  # ROOT / mouseID / day / session
LEVEL_SELECTORS = None  # e.g. {"l0_selector": ["your-mouse"]}
SOURCES = ("trials", "events", "dlc")

INDIVIDUAL = "individual_0"
TRACKING_KEYPOINT = "body"  # one tracked point, used consistently for paths/speed/occupancy
LEFT_EAR, RIGHT_EAR = "lear", "rear"
BODY_FRONT, BODY_BACK = "body", "tailbase"
CAMERA_VIEW = "top_down"  # DLC image coordinates: +x right, +y down
REFERENCE_VECTOR = (1.0, 0.0)  # fixed axis; register to arena axes if different

CONFIDENCE_THRESHOLD = 0.9  # example setting: inspect your own confidence distribution
MAX_GAP_FRAMES = None  # None disables interpolation; an integer fills only short gaps
SMOOTHING_WINDOW = None  # None disables movement's rolling median filter

TRIAL_ROWS = (0, 1, 2)  # zero-based rows of the displayed, time-sorted session trial table
WINDOW = None  # absolute seconds (start, end); None shows the first 60 seconds of trials
OCCUPANCY_BINS = 40
ARENA_RANGE = None  # optionally ((xmin, xmax), (ymin, ymax)), in original pixels

config = build_config(
    ROOT, session=SESSION, level_names=LEVEL_NAMES,
    level_selectors=LEVEL_SELECTORS, sources=SOURCES,
)

## Load, inspect coverage, and process

In [ ]:
# Preview rejects zero or multiple selected sessions before any source data is loaded.
display(preview_session(config))
session_data = load_session(config, individual=INDIVIDUAL)
display(session_data.loaded.stream_coverage)
raw_pose = session_data.raw_pose
pose = prepare_pose(
    raw_pose, confidence_threshold=CONFIDENCE_THRESHOLD,
    max_gap_frames=MAX_GAP_FRAMES, smoothing_window=SMOOTHING_WINDOW,
)
position = pose.position.sel(individual=INDIVIDUAL, drop=True)
if TRACKING_KEYPOINT not in position.keypoint:
    raise ValueError(f"Choose TRACKING_KEYPOINT from {position.keypoint.values.tolist()}")
point = position.sel(keypoint=TRACKING_KEYPOINT, drop=True)
point.attrs["units"] = "px"
trials = session_data.trials.sort_values("start_time").reset_index(drop=True)
events = session_data.events
if trials.empty:
    raise ValueError("No parsed trials are available in this session.")
display(trials)
for note in qc_annotations(trials, events).notes:
    print(note)
print("Keypoints:", position.keypoint.values.tolist())
print("Valid tracked-point frames:", int(point.notnull().all("space").sum()),
      "/", point.sizes["time"])

if WINDOW is None:
    window_start = max(float(point.time.min()), float(trials.start_time.iloc[0]))
    WINDOW = (window_start, min(window_start + 60, float(point.time.max())))
if not np.isfinite(WINDOW).all() or WINDOW[0] >= WINDOW[1]:
    raise ValueError("WINDOW must be a finite increasing interval overlapping the pose.")
if not bool(((point.time >= WINDOW[0]) & (point.time <= WINDOW[1])).any()):
    raise ValueError("WINDOW contains no recorded pose samples; choose a window on the session clock.")
selected_rows = [i for i in TRIAL_ROWS if 0 <= i < len(trials)]
if len(selected_rows) != len(TRIAL_ROWS):
    print("Trial rows outside this session were omitted:", sorted(set(TRIAL_ROWS) - set(selected_rows)))
if not selected_rows:
    raise ValueError("TRIAL_ROWS must select at least one displayed trial row.")

## Single-trial and multi-trial trajectories

Every path uses `movement.plots.plot_centroid_trajectory` with the single
configured keypoint. This avoids a changing centroid when different keypoints
are missing. The single-trial view is coloured by absolute time; the overlay
uses one colour per trial. Axes retain the DLC image convention (+y down).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), layout="constrained")
trial_row = selected_rows[0]
trial = trials.iloc[trial_row]
trial_point = point.sel(time=slice(float(trial.start_time), float(trial.end_time)))
if trial_point.sizes["time"] < 2 or not bool(trial_point.notnull().all("space").any()):
    raise ValueError(f"Trial row {trial_row} has insufficient tracked positions; choose another row.")
plot_centroid_trajectory(trial_point, ax=axes[0], s=7, cmap="viridis")
axes[0].set_title(f"Single trajectory — trial row {trial_row}")
for index, trial_row in enumerate(selected_rows):
    trial = trials.iloc[trial_row]
    trial_point = point.sel(time=slice(float(trial.start_time), float(trial.end_time)))
    if trial_point.sizes["time"] < 2 or not bool(trial_point.notnull().all("space").any()):
        print(f"No trajectory plotted for trial row {trial_row}: insufficient valid positions")
        continue
    plot_centroid_trajectory(trial_point, ax=axes[1], c=f"C{index % 10}",
                             s=7, alpha=0.6, label=f"Trial row {trial_row}")
axes[1].set_title("Multiple trial trajectories")
axes[1].legend()
for ax in axes:
    ax.set_aspect("equal")
    ax.invert_yaxis()
    ax.set_xlabel("x (px)")
    ax.set_ylabel("y (px)")
plt.show()

## Inbound deviation and outbound straightness

A target-zone trigger separates outbound and inbound. Trials without a known
trigger have no defensible phase boundary here and receive no phase metric.

**Inbound deviation:** `compute_path_deviation` returns unsigned perpendicular
distance from each position to the infinite line through the first and last
valid positions of that inbound path. This is observed-path geometry; it is not
error relative to a configured home port or the intended return direction.

**Outbound complexity:** use the built-in **path straightness**, D/L (endpoint
distance / travelled distance). One means straight; lower values mean less
direct. This is a specific measure of directness, not a general measure of
complexity, tortuosity or loop count. It depends on sampling and filtering.

The table records exclusions explicitly. Metrics require at least two samples
and complete finite tracked-point coordinates after the configured processing,
so the path-length implementation cannot fill remaining NaN gaps. This does
not detect missing timestamps: inspect the acquisition clock for long intervals.
Phases outside the available pose clock are excluded. No trial metrics are
computed after concatenating trials. The reported mean deviation is an
arithmetic mean across recorded samples, not a time-weighted mean.

In [ ]:
records = []
inbound_deviations = {}
for trial_row, trial in trials.iterrows():
    start, split, end = (float(trial.start_time), float(trial.tz_triggered_time),
                         float(trial.end_time))
    for phase, bounds in [("outbound", (start, split)), ("inbound", (split, end))]:
        record = {"trial_row": trial_row, "phase": phase, "outcome": trial.outcome,
                  "duration_s": np.nan, "path_straightness": np.nan,
                  "mean_deviation_px": np.nan, "status": ""}
        if not np.isfinite([start, split, end]).all() or not start < split < end:
            record["status"] = "no valid target-trigger phase boundary"
        elif bounds[0] < float(point.time.min()) or bounds[1] > float(point.time.max()):
            record["status"] = "incomplete pose time coverage"
        else:
            record["duration_s"] = bounds[1] - bounds[0]
            segment = point.sel(time=slice(*bounds))
            if segment.sizes["time"] < 2:
                record["status"] = "fewer than two pose samples"
            elif not bool(np.isfinite(segment).all()):
                record["status"] = "unfilled tracking gaps"
            elif phase == "outbound":
                value = float(kin.compute_path_straightness(segment))
                record["path_straightness"] = value
                record["status"] = "ok" if np.isfinite(value) else "undefined: zero path length"
            elif bool((segment.isel(time=0) == segment.isel(time=-1)).all()):
                record["status"] = "undefined: coincident endpoints"
            else:
                deviation = kin.compute_path_deviation(segment)
                inbound_deviations[trial_row] = deviation
                record["mean_deviation_px"] = float(deviation.mean("time"))
                record["status"] = "ok"
        records.append(record)
metrics = pd.DataFrame(records)
display(metrics)
display(metrics.groupby(["phase", "status"]).size().rename("n_trials"))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), layout="constrained")
for trial_row in selected_rows:
    if trial_row not in inbound_deviations:
        continue
    deviation = inbound_deviations[trial_row]
    phase_start = float(trials.iloc[trial_row].tz_triggered_time)
    axes[0].plot(deviation.time - phase_start, deviation, label=f"Trial row {trial_row}")
axes[0].set(xlabel="Time from target trigger (s)", ylabel="Inbound deviation (px)",
            title="Distance to observed endpoint line")
if axes[0].lines:
    axes[0].legend()
else:
    axes[0].text(0.5, 0.5, "Selected trials have no valid inbound metric", ha="center",
                 transform=axes[0].transAxes)

outbound = metrics[(metrics.phase == "outbound") & (metrics.status == "ok")]
axes[1].scatter(outbound.duration_s, outbound.path_straightness, s=22)
axes[1].set(xlabel="Outbound duration (s)", ylabel="Outbound straightness (D/L)",
            ylim=(-0.02, 1.02), title="Outbound directness by duration")
paired = metrics.pivot(index="trial_row", columns="phase",
                       values=["path_straightness", "mean_deviation_px"])
axes[2].scatter(paired[("path_straightness", "outbound")],
                paired[("mean_deviation_px", "inbound")], s=22)
axes[2].set(xlabel="Outbound straightness (D/L)", ylabel="Mean inbound deviation (px)",
            title="Within-trial outbound / inbound comparison")
plt.show()

API references: [trajectory](https://movement.neuroinformatics.dev/latest/api/movement.plots.plot_centroid_trajectory.html),
[path straightness](https://movement.neuroinformatics.dev/latest/api/movement.kinematics.compute_path_straightness.html),
[path deviation](https://movement.neuroinformatics.dev/latest/api/movement.kinematics.compute_path_deviation.html).
Sample means and panel arrangement are ordinary xarray/Matplotlib summaries of these outputs.